# Hausse de loyer 2026 - Collection Équinoxe

**Question :** de combien les loyers de Collection Équinoxe augmenteront-ils en 2026?

| | |
|---|---|
| **Définition retenue** | croissance annualisée du **loyer effectif** (`sRentEffective`) **à unité constante** : moyenne des hausses prévues par appartement, renouvellements et relocations pondérés, pour les six immeubles de l’extrait |
| **Estimation 2026** | **5,44 %** (scénarios : 4,93 % pour seuls renouvellements; 6,24 % pour seules relocations) |
| **Méthode** | régression linéaire entraînée sur 2 246 paires exploitables jusqu’en 2025, avec trois facteurs publics décalés d’un an |
| **Validation** | test 2025 : MAE 4,22 points, RMSE 5,97 points, R² 0,015; backtests 2023–2025 présentés en section 7 |

### Plan, aligné sur les critères d'évaluation

| Section | Critère des consignes |
|---|---|
| 1. Définir la hausse | Définition de la hausse |
| 2. Analyser les données | Analyse des données et effet de mix |
| 3. Mesurer à unité constante | Appariement à unité constante |
| 4. Concessions | Concessions |
| 5. Renouvellements, relocations et Ontario | Renouvellements vs relocations (+ bonus Ontario) |
| 6. Données publiques | Données externes |
| 7. Prévoir et valider | Prévision et backtest (+ bonus : par immeuble, par chambres, fourchette, tableau de bord) |
| 8. Hypothèses et limites, 9. Références | |

## 0. Configuration et chargement

Toutes les constantes sont déclarées ici, une seule fois, avec leur justification : aucun nombre n'apparaît « en dur » plus bas.
Les fichiers sources ne sont jamais modifiés; toute transformation se fait en mémoire.

In [ ]:
import os
import sys
from pathlib import Path

os.environ.setdefault("LOKY_MAX_CPU_COUNT", str(os.cpu_count()))   # évite un avertissement de joblib sous Windows
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.insert(0, "public_data")
from extract_cmhc import MARKETS as CMHC_MARKET_LABELS, extract_public_data, number, read_workbook

pd.set_option("display.width", 180)
pd.set_option("display.max_columns", 40)

# --- Fichiers (lecture seule) ---
PRIVATE_DIR = Path("private_data")
RAW_PUBLIC_DIR = Path("raw_public_data")
LISTINGS_PATH = PRIVATE_DIR / "equinoxe_listings.csv"
LEASES_PATH = PRIVATE_DIR / "equinoxe_lease_history.csv"
CONCESSIONS_PATH = PRIVATE_DIR / "equinoxe_concessions.csv"
ASKING_PATH = PRIVATE_DIR / "equinoxe_asking_history.csv"
REGULATORY_RATES_PATH = Path("public_data") / "regulatory_rates.csv"
CPI_PATH = RAW_PUBLIC_DIR / "IPC.csv"
LABOUR_PATH = RAW_PUBLIC_DIR / "msrche-du-travail.csv"
DEMOGRAPHY_PATH = RAW_PUBLIC_DIR / "evolution-demographique.csv"

# --- Clé d'unité ---
UNIT_KEY = ["sPropCode", "sUnitCode"]       # 1 pour 1 avec hUnit et sLink (section 2a)
WRONG_UNIT_KEY = ["sSite", "sUnitCode"]     # clé interdite, utilisée une seule fois pour montrer l'erreur (3b)

# --- Appariement ---
MIN_GAP_YEARS = 0.5        # paires plus rapprochées : issues d'un bail de 6 mois, non comparables à un bail annuel (3a)
MAX_GAP_YEARS = 2.5        # paires plus espacées : l'unité a pu être rénovée ou rester vide longtemps
GAP_BINS_YEARS = [0, MIN_GAP_YEARS, 0.95, 1.05, 1.6, MAX_GAP_YEARS, np.inf]   # tranches d'écart pour décrire les paires
DAYS_PER_YEAR = 365.25
PCT = 100

# --- Codes et libellés ---
RENEWAL, TURNOVER = 1, 0                  # sRenewal : 1 = locataire en place, 0 = relocation
QUEBEC, ONTARIO = "Quebec", "Ontario"     # valeurs de sState
PROMO_CHARGE_CODE = "PromoPay"
TWO_BEDROOMS = 2

# --- Périodes ---
FIRST_ANALYSIS_YEAR = 2019        # avant 2019, trop peu de paires de renouvellement
LAST_OBSERVED_YEAR = 2025         # les données s'arrêtent au 2025-12-31
FORECAST_YEAR = LAST_OBSERVED_YEAR + 1
REQUIRED_BACKTEST_YEARS = [2023, 2024, 2025]                      # exigées par les consignes
EXTENDED_BACKTEST_YEARS = [2021, 2022] + REQUIRED_BACKTEST_YEARS  # 3 années ne suffisent pas à départager des variantes
FIRST_PUBLIC_YEAR = 2020          # SCHL : le fichier 2021 donne aussi la variation 2020; les autres séries commencent en 2021

# --- Tolérances et paramètres d'analyse ---
RENT_MATCH_TOLERANCE = 1.0        # $ / mois : deux loyers concordent à un arrondi au dollar près
SIX_MONTH_GAP_YEARS = 0.4         # seuil alternatif qui inclut les baux de 6 mois (sensibilité, 3d)
TRIM_SHARE = 0.05                 # moyenne tronquée : retire 5 % des paires de chaque côté (sensibilité, 3d)
ONTARIO_EXEMPTION_DATE = pd.Timestamp("2018-11-15")   # première occupation après cette date : exempté de la ligne directrice



### Chargement

On charge les quatre tables et on convertit les dates une seule fois. `lease_start_year` (année de `sLeaseFrom`) est l'année
à laquelle on rattache un bail : c'est la date où le nouveau loyer **entre en vigueur**, donc celle qui compte pour les revenus.

In [ ]:
listings = pd.read_csv(LISTINGS_PATH)
leases = pd.read_csv(LEASES_PATH, parse_dates=["sLeaseFrom", "sLeaseTo", "sSignDate", "sAvailable"])
concessions = pd.read_csv(CONCESSIONS_PATH, parse_dates=["sDateFrom", "sDateTo"])
asking = pd.read_csv(ASKING_PATH)

leases["lease_start_year"] = leases["sLeaseFrom"].dt.year
asking["ask_month"] = pd.PeriodIndex(asking["sMonth"], freq="M").to_timestamp()

for name, table in [("listings", listings), ("leases", leases), ("concessions", concessions), ("asking", asking)]:
    print(f"{name:<12} {len(table):>6,} lignes  {table.shape[1]:>2} colonnes")
print(f"\nbaux du {leases['sLeaseFrom'].min():%Y-%m-%d} au {leases['sLeaseFrom'].max():%Y-%m-%d}")

## 1. Définir la hausse mesurée

« La hausse de loyer 2026 » n'est pas un seul chiffre. Les candidats (chiffrés en section 3c) :

| Définition candidate | Ce qu'elle mesure |
|---|---|
| Médiane annuelle de tous les baux | le loyer du bail « typique » de l'année : **mesure surtout le mix d'immeubles** (section 2f) |
| Même unité, loyer contractuel (`sRent`) | la hausse du prix inscrit au bail, pour un même appartement |
| **Même unité, loyer effectif (`sRentEffective`)** | **la hausse de l'argent réellement perçu, pour un même appartement** |
| Renouvellements seulement | la hausse consentie aux locataires en place (TAL au Québec, ligne directrice en Ontario) |
| Relocations seulement | la hausse du prix de marché quand une unité change de locataire |

> **Définition retenue : hausse 2026 = croissance annualisée du loyer effectif à unité constante, moyenne des prévisions par appartement de l’extrait, après pondération des scénarios renouvellement et relocation, pour les six immeubles.**

**Pourquoi :**

1. **À unité constante.** La médiane annuelle bondit de 10,8 % en 2023 alors qu'un même appartement augmente d'environ 2 % :
   elle mesure l'arrivée de trois immeubles haut de gamme, pas une hausse payée par les locataires (2f, 3c).
2. **Loyer effectif.** La question d'affaires est de **budgéter les revenus**, c'est-à-dire ce qui est perçu après les mois
   gratuits et autres concessions. Depuis 2024, les deux mesures divergent : en 2025, +7,0 % au bail mais +3,6 % perçu (section 4).
   Citer le contractuel surestimerait la hausse des revenus de plus de trois points.
3. **Tous les baux, mais modélisés selon leur type.** Les revenus 2026 viendront des renouvellements et des relocations, donc le
   chiffre les combine. Comme ils obéissent à des règles différentes (section 5), le modèle connaît le type de bail et
   pondère les deux cas.
4. **Moyenne des prévisions, croissance annualisée.** Chaque appartement reçoit le même poids; les scénarios sont pondérés selon le type de bail. Les médianes historiques restent des indicateurs descriptifs robustes, distincts de cette moyenne prévue.

**Indicateurs secondaires** rapportés à côté : renouvellements en loyer contractuel (ce que le TAL et l'Ontario encadrent),
relocations (référence pour les loyers affichés), prévision par immeuble et par nombre de chambres.

## 2. Analyser les données

Les descriptions ne correspondent pas toujours aux données. Chaque sous-section teste une hypothèse et se termine par une
**lecture** et une **conséquence**. On garde les noms Yardi pendant les vérifications, puis on renomme en 2e.

### 2a. La clé d'unité

In [ ]:
unit_key_checks = pd.Series({
    "hUnit distincts par sPropCode + sUnitCode (max)": leases.groupby(UNIT_KEY)["hUnit"].nunique().max(),
    "clés sPropCode + sUnitCode par hUnit (max)": leases.groupby("hUnit")[UNIT_KEY].nunique().max().max(),
    "clés sPropCode + sUnitCode par sLink (max)": leases.groupby("sLink")[UNIT_KEY].nunique().max().max(),
    "unités en collision sur sSite + sUnitCode": listings.duplicated(WRONG_UNIT_KEY, keep=False).sum(),
    "unités de listings": len(listings),
    "baux en double (même unité, même début)": leases.duplicated(UNIT_KEY + ["sLeaseFrom"]).sum(),
})
unit_key_checks.to_frame("valeur")

**Lecture :** `sPropCode + sUnitCode`, `hUnit` et `sLink` sont trois clés équivalentes (1 pour 1). `sSite + sUnitCode`
fusionnerait 248 appartements distincts : un site regroupe plusieurs propriétés dont les numéros d'unité se chevauchent.
Aucun doublon.

**Conséquence :** tout appariement se fait sur `UNIT_KEY = ["sPropCode", "sUnitCode"]`.

### 2b. Ce que représentent les colonnes des baux

In [ ]:
first_unit_leases = leases[leases["sTermSeq"] == 1]
lease_column_checks = pd.Series({
    "sAvailable == sLeaseFrom (% des baux)": (leases["sAvailable"] == leases["sLeaseFrom"]).mean() * PCT,
    "baux avec sTermSeq = 1": len(first_unit_leases),
    "unités distinctes": leases.groupby(UNIT_KEY).ngroups,
    "sTermSeq = 1 : % de relocations": first_unit_leases["sRenewal"].eq(TURNOVER).mean() * PCT,
    "sConcession == (sRentEffective < sRent) (% des baux)":
        (leases["sConcession"] == (leases["sRentEffective"] < leases["sRent"])).mean() * PCT,
}).round(1)
print(lease_column_checks.to_string(), end="\n\n")
print("Étiquette sLeaseTerm x durée réelle sTermMonths :")
pd.crosstab(leases["sLeaseTerm"], leases["sTermMonths"].round())

**Lecture :**

| Colonne | Constat | Ce qu'elle est réellement |
|---|---|---|
| `sAvailable` | égale à `sLeaseFrom` pour 100 % des baux | une copie de la date de début |
| `sTermSeq` | autant de `sTermSeq = 1` que d'unités, tous des relocations | le **rang du bail dans l'historique de l'unité**, pas du locataire |
| `sConcession` | vaut 1 exactement quand `sRentEffective < sRent` | un indicateur **dérivé** des deux loyers |
| `sTermMonths` | surtout 12 mois, parfois 6, 18 ou 24 | une durée calculée à partir des dates, fiable |
| `sLeaseTerm` | « 6 Months » ou « Short Term » contiennent des baux de 12 mois | une étiquette commerciale **non fiable** |

**Conséquences :** le type de bail vient de `sRenewal`, jamais de `sTermSeq`; `sConcession` n'est jamais utilisé à côté du
loyer effectif (même information deux fois); les hausses sont **annualisées** avec les dates réelles; `sAvailable` et
`sLeaseTerm` sont ignorés.

### 2c. Le piège `PromoPay`

La table des concessions décrit `PromoPay` comme un montant **mensuel**. On rattache chaque concession au bail de la même unité
dont la période contient son début, puis on exprime son montant en mois de loyer.

In [ ]:
def attach_concessions_to_leases(concession_df, lease_df):
    """Rattache chaque concession au bail de la même unité dont la période contient sDateFrom."""
    lease_columns = UNIT_KEY + ["sLeaseFrom", "sLeaseTo", "sRent"]
    candidates = (concession_df.reset_index(names="concession_id")
                  .merge(lease_df[lease_columns].reset_index(names="lease_id"), on=UNIT_KEY))
    starts_within_lease = candidates["sDateFrom"].between(candidates["sLeaseFrom"], candidates["sLeaseTo"])
    return candidates[starts_within_lease]


matched_concessions = attach_concessions_to_leases(concessions, leases)
concessions_by_code = (concessions.groupby("sChargeCode")
                       .agg(concessions=("sAmount", "size"),
                            median_amount=("sAmount", "median"),
                            median_months=("sMonths", "median"),
                            share_single_month=("sMonths", lambda months: (months == 1).mean()))
                       .sort_values("concessions", ascending=False))
print(f"concessions rattachées à un bail : {len(matched_concessions):,} / {len(concessions):,}\n")
print(concessions_by_code.round(2), end="\n\n")

promo = matched_concessions[matched_concessions["sChargeCode"] == PROMO_CHARGE_CODE]
promo_in_months_of_rent = -promo["sAmount"] / promo["sRent"]

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.hist(promo_in_months_of_rent, bins=40)
ax.set_xlabel("montant PromoPay / loyer mensuel du bail (en mois de loyer)")
ax.set_ylabel("concessions")
ax.set_title("PromoPay : un montant unique proche d'un mois de loyer, pas un rabais mensuel")
plt.tight_layout()
plt.show()

**Lecture :** `PromoPay` est inscrit sur **un seul mois** dans 88 % des cas, pour environ **un mois de loyer**. S'il était mensuel,
les résidents habiteraient presque gratuitement : la description est fausse. C'est un **mois gratuit unique à la signature**.
Les autres codes valent environ 130 à 150 $ par mois sur 3 à 12 mois; `freepark`, `freelock` et `freeappl` portent sur des
**accessoires**, pas sur le loyer.

**Conséquence :** `PromoPay` est renommé `one_time_signing_discount`; son effet doit être **étalé sur la durée du bail**
(un mois gratuit sur 12 mois ≈ 8 % de rabais). On vérifie que `sRentEffective` fait exactement cela.

### 2d. Validation de `sRentEffective`

Hypothèse : loyer effectif = loyer contractuel + (somme des concessions du bail, `sAmount × sMonths`) / durée du bail.

In [ ]:
concession_totals = (matched_concessions
                     .assign(total_amount=matched_concessions["sAmount"] * matched_concessions["sMonths"])
                     .groupby("lease_id")
                     .agg(concession_total=("total_amount", "sum"), concession_count=("total_amount", "size")))
reconstruction = leases.join(concession_totals).fillna({"concession_total": 0, "concession_count": 0})
observed_gap = reconstruction["sRentEffective"] - reconstruction["sRent"]
predicted_gap = reconstruction["concession_total"] / reconstruction["sTermMonths"]
reconstruction["is_reconstructed"] = (observed_gap - predicted_gap).abs() < RENT_MATCH_TOLERANCE

print(f"sRentEffective reconstruit au dollar près : {reconstruction['is_reconstructed'].mean():.1%} des baux\n")
failures = reconstruction[~reconstruction["is_reconstructed"]]
pd.crosstab(observed_gap[failures.index] < 0, failures["concession_count"] > 0,
            rownames=["rabais observé"], colnames=["concessions dans la table"])

**Lecture :** la formule reconstruit `sRentEffective` **au dollar près pour environ 62 % des baux** : une concordance exacte sur
des milliers de baux n'est pas une coïncidence. `sRentEffective` est bien le loyer contractuel diminué des concessions étalées
sur le bail, `PromoPay` compris. Les échecs viennent de la **table des concessions** : rabais sans concession dans l'extrait
(table incomplète), concessions sans rabais (surtout des accessoires non déduits du loyer), rattachement imparfait.

**Conséquence :** on utilise `sRentEffective` tel que calculé par Yardi, plutôt que de le recalculer à partir d'une table incomplète.

### 2e. Renommage des colonnes

**Cette étape modifie les tables** : à partir d'ici, les colonnes portent le nom de ce qu'elles contiennent réellement (2b à 2d).
Les clés et les colonnes descriptives (`sPropCode`, `sUnitCode`, `sBuilding`, `sState`, `sBeds`...) gardent leur nom Yardi, déjà exact.

In [ ]:
LEASE_COLUMN_RENAMES = {
    "sRent": "contract_rent",
    "sRentEffective": "effective_rent",            # contractuel - concessions étalées sur le bail (2d)
    "sConcession": "has_concession",               # = effective_rent < contract_rent (2b)
    "sRenewal": "is_renewal",                      # 1 = locataire en place, 0 = relocation
    "sTermSeq": "unit_lease_rank",                 # rang du bail dans l'historique de l'unité (2b)
    "sTermMonths": "lease_term_months",
    "sLeaseFrom": "lease_start",
    "sLeaseTo": "lease_end",
    "sSignDate": "sign_date",
    "sAvailable": "lease_start_duplicate",         # copie exacte de sLeaseFrom (2b)
    "sLeaseTerm": "lease_term_label_unreliable",   # étiquette incohérente avec la durée réelle (2b)
}
LISTING_COLUMN_RENAMES = {"sRent": "proposed_next_rent"}     # loyer proposé au prochain bail (section 7g)
CONCESSION_COLUMN_RENAMES = {"sAmount": "amount_per_month", "sMonths": "months_applied",
                             "sDateFrom": "concession_start", "sDateTo": "concession_end"}
CHARGE_CODE_LABELS = {
    "PromoPay": "one_time_signing_discount",       # PAS mensuel : environ un mois de loyer, une fois (2c)
    "freerent": "free_rent_monthly",
    "freepark": "free_parking_accessory",
    "freelock": "free_locker_accessory",
    "freeappl": "free_appliances_accessory",
    "Freeothe": "other_uncategorized",
    "Indem": "tenant_indemnity",
    "Referenc": "referral_credit",
}

leases = leases.rename(columns=LEASE_COLUMN_RENAMES)
listings = listings.rename(columns=LISTING_COLUMN_RENAMES)
concessions = concessions.rename(columns=CONCESSION_COLUMN_RENAMES)
concessions["charge_type"] = concessions["sChargeCode"].map(CHARGE_CODE_LABELS)
leases[list(LEASE_COLUMN_RENAMES.values())].head(3)

### 2f. L'effet de composition (mix)

Pourquoi la médiane annuelle de tous les baux ne mesure pas la hausse : on regarde **qui** est loué chaque année.

In [ ]:
analysis_window = leases[leases["lease_start_year"].between(FIRST_ANALYSIS_YEAR, LAST_OBSERVED_YEAR)].copy()
analysis_window["rent_per_sqft"] = analysis_window["contract_rent"] / analysis_window["sSqft"]

mix_by_year = analysis_window.groupby("lease_start_year").agg(
    leases=("contract_rent", "size"),
    buildings=("sBuilding", "nunique"),
    median_rent=("contract_rent", "median"),
    median_sqft=("sSqft", "median"),
    pct_two_bed_plus=("sBeds", lambda beds: (beds >= TWO_BEDROOMS).mean() * PCT),
    median_rent_per_sqft=("rent_per_sqft", "median"))
mix_by_year["naive_median_growth_pct"] = mix_by_year["median_rent"].pct_change() * PCT
print(mix_by_year.round(2))

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
share_by_building = pd.crosstab(analysis_window["lease_start_year"], analysis_window["sBuilding"], normalize="index") * PCT
share_by_building.plot(kind="bar", stacked=True, ax=axes[0], width=0.85)
axes[0].set(xlabel="année de début du bail", ylabel="% des baux de l'année",
            title="Trois immeubles haut de gamme entrent en 2023-2024")
axes[0].legend(fontsize=8, loc="upper left", bbox_to_anchor=(1, 1))
analysis_window.groupby("sBuilding")["rent_per_sqft"].median().sort_values().plot(kind="barh", ax=axes[1])
axes[1].set(xlabel="loyer contractuel médian au pi² ($)", ylabel="",
            title="Les nouveaux immeubles sont beaucoup plus chers au pi²")
plt.tight_layout()
plt.show()

**Lecture :** jusqu'en 2022, seuls les trois immeubles de Laval génèrent des baux. Le Carlyle et The Met entrent en 2023,
Westpark en 2024. Les unités louées deviennent **plus petites** et comptent moins de 2 chambres et plus, ce qui devrait faire
*baisser* la médiane, et pourtant elle bondit de **10,8 % en 2023** : les nouveaux immeubles sont beaucoup plus chers au pi².
La médiane mélange des produits différents.

**Conséquence :** la médiane annuelle mesure la **composition du portefeuille**, pas la hausse. On compare chaque unité à
elle-même (section 3). **Limite :** avant 2023, l'historique ne couvre que Laval; le backtest 2023 prédit donc Mont-Royal et
Ottawa avec un historique lavallois.

## 3. Mesurer la croissance à unité constante

Trois étapes : **apparier** chaque bail au bail précédent de la même unité (`UNIT_KEY`), **filtrer** les paires non
comparables, **annualiser** :

$$\text{croissance annualisée} = \left(\frac{\text{loyer}}{\text{loyer précédent}}\right)^{1/\text{écart en années}} - 1$$

La puissance (plutôt qu'une division) tient compte de la composition des hausses : 1 000 $ → 1 060 $ en 18 mois donne
3,96 %/an, et non 6 %. Pour chaque paire, on calcule aussi le **rabais du bail précédent** et la **hausse du rabais**
(section 4), et on range la paire dans un **segment** : renouvellement au Québec, renouvellement en Ontario, ou relocation (section 5).

### 3a. Apparier, filtrer, annualiser

In [ ]:
RENT_COLUMNS = ["contract_rent", "effective_rent"]
PREVIOUS_LEASE_COLUMNS = RENT_COLUMNS + ["lease_start"]


def pair_consecutive_leases(lease_df, unit_key=UNIT_KEY):
    """Associe chaque bail au bail précédent de la même unité (colonnes `prev_`); le premier bail d'une unité n'a pas de paire."""
    ordered = lease_df.sort_values(unit_key + ["lease_start"]).copy()
    previous = ordered.groupby(unit_key)[PREVIOUS_LEASE_COLUMNS].shift(1).add_prefix("prev_")
    paired = pd.concat([ordered, previous], axis=1).dropna(subset=["prev_lease_start"])
    paired["gap_years"] = (paired["lease_start"] - paired["prev_lease_start"]).dt.days / DAYS_PER_YEAR
    return paired


def is_comparable(paired):
    """Écart entre MIN_GAP_YEARS et MAX_GAP_YEARS, et quatre loyers positifs."""
    has_comparable_gap = paired["gap_years"].between(MIN_GAP_YEARS, MAX_GAP_YEARS, inclusive="neither")
    rent_columns = RENT_COLUMNS + [f"prev_{rent}" for rent in RENT_COLUMNS]
    return has_comparable_gap & paired[rent_columns].gt(0).all(axis=1)


def annualized_growth_pct(paired, rent_column):
    ratio = paired[rent_column] / paired[f"prev_{rent_column}"]
    return (ratio ** (1 / paired["gap_years"]) - 1) * PCT


def discount_pct(effective_rent, contract_rent):
    """Part du loyer contractuel qui n'est pas perçue (%)."""
    return (1 - effective_rent / contract_rent) * PCT


def build_comparable_pairs(lease_df):
    """Paires comparables avec croissances annualisées, rabais du bail précédent, hausse du rabais et segment."""
    paired = pair_consecutive_leases(lease_df)
    pairs = paired[is_comparable(paired)].copy()
    for rent in RENT_COLUMNS:
        pairs[f"{rent}_growth_pct"] = annualized_growth_pct(pairs, rent)
    pairs["prev_discount_pct"] = discount_pct(pairs["prev_effective_rent"], pairs["prev_contract_rent"])
    pairs["discount_change_pts"] = discount_pct(pairs["effective_rent"], pairs["contract_rent"]) - pairs["prev_discount_pct"]
    is_renewal, is_ontario = pairs["is_renewal"].eq(RENEWAL), pairs["sState"].eq(ONTARIO)
    pairs["segment"] = np.select([is_renewal & ~is_ontario, is_renewal & is_ontario],
                                 ["renewal_QC", "renewal_ON"], default="turnover")
    return pairs


all_pairs = pair_consecutive_leases(leases)
comparable_pairs = build_comparable_pairs(leases)
pairs = comparable_pairs[comparable_pairs["lease_start_year"] >= FIRST_ANALYSIS_YEAR]

all_pairs["gap_bucket"] = pd.cut(all_pairs["gap_years"], bins=GAP_BINS_YEARS)
all_pairs["raw_contract_growth_pct"] = (all_pairs["contract_rent"] / all_pairs["prev_contract_rent"] - 1) * PCT
print(pd.Series({
    "baux": len(leases),
    "paires (bail précédent trouvé)": len(all_pairs),
    "paires comparables (filtres)": len(comparable_pairs),
    f"paires comparables depuis {FIRST_ANALYSIS_YEAR}": len(pairs),
}).to_string(), end="\n\n")
all_pairs.groupby("gap_bucket", observed=True).agg(
    pairs=("gap_years", "size"), median_raw_contract_growth_pct=("raw_contract_growth_pct", "median")).round(2)

**Lecture :** la grande majorité des paires sont espacées d'**un an**, le cas d'un bail de 12 mois suivi du suivant.
La hausse **brute** est à peu près la même quel que soit l'écart (environ 4,5 %) : le loyer augmente **par bail**, pas au
prorata du temps. Annualiser ramène donc les paires de 18-24 mois à leur vrai rythme annuel, ce qui compte pour un budget annuel.
Le filtre retire les 62 paires de moins de 6 mois (baux de 6 mois, dont la hausse annualisée doublerait artificiellement);
aucune paire ne dépasse 2,5 ans, mais le seuil protège la méthode.

### 3b. Ce que ferait la mauvaise clé (`sSite + sUnitCode`)

On apparie **une seule fois** sur la clé interdite, pour montrer l'effet.

In [ ]:
wrong_key_pairs = pair_consecutive_leases(leases, WRONG_UNIT_KEY)
raw_growth = lambda paired: (paired["contract_rent"] / paired["prev_contract_rent"] - 1) * PCT
pd.DataFrame({
    "sPropCode + sUnitCode": raw_growth(all_pairs).describe(),
    "sSite + sUnitCode (interdit)": raw_growth(wrong_key_pairs).describe(),
}).round(2)

**Lecture :** la mauvaise clé crée des paires qui n'existent pas et compare un appartement à un **autre** appartement du même
site : on voit apparaître des « baisses » et des « hausses » absurdes, et un écart-type plusieurs fois plus grand. Avec la bonne
clé, toutes les variations restent dans une plage plausible.

### 3c. Résultats : les définitions candidates chiffrées

In [ ]:
def median_by_year(pair_df, column, mask=None):
    selected = pair_df if mask is None else pair_df[mask]
    return selected.groupby("lease_start_year")[column].median()


analysis_years = list(range(FIRST_ANALYSIS_YEAR, LAST_OBSERVED_YEAR + 1))
candidate_definitions = pd.DataFrame({
    "naive_median_all_leases": mix_by_year["naive_median_growth_pct"],
    "same_unit_contract": median_by_year(pairs, "contract_rent_growth_pct"),
    "same_unit_effective": median_by_year(pairs, "effective_rent_growth_pct"),
    "renewal_effective": median_by_year(pairs, "effective_rent_growth_pct", pairs["is_renewal"].eq(RENEWAL)),
    "turnover_effective": median_by_year(pairs, "effective_rent_growth_pct", pairs["is_renewal"].eq(TURNOVER)),
    "pairs": pairs.groupby("lease_start_year").size(),
}).loc[analysis_years]
yearly_target = candidate_definitions["same_unit_effective"]   # la cible (section 1)
print(candidate_definitions.round(2))

fig, ax = plt.subplots(figsize=(10, 4.5))
for column, style, label in [("naive_median_all_leases", "o:", "médiane naïve de tous les baux"),
                             ("same_unit_contract", "o-", "même unité, loyer contractuel"),
                             ("same_unit_effective", "s-", "même unité, loyer effectif (définition retenue)")]:
    ax.plot(analysis_years, candidate_definitions[column], style, lw=2, label=label)
ax.axhline(0, color="grey", lw=0.8)
ax.set(xlabel="année de début du nouveau bail", ylabel="croissance annuelle (%)",
       title="La médiane naïve mesure le mix; contractuel et effectif divergent depuis 2024")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

**Lecture :** la hausse réelle d'un même appartement est bien plus modeste que la médiane naïve (2023 : +10,8 % naïf contre
environ 2 % à unité constante). Contractuel et effectif évoluent ensemble jusqu'en 2023, puis divergent (section 4).
La cible oscille entre 1,6 % et 4,6 % sur 2019-2025, avec un creux en 2023-2024.

### 3d. Sensibilité aux choix de méthode

On recalcule la croissance effective en changeant **un choix à la fois**.

In [ ]:
all_pairs["raw_effective_growth_pct"] = (all_pairs["effective_rent"] / all_pairs["prev_effective_rent"] - 1) * PCT
all_pairs["annualized_effective_growth_pct"] = annualized_growth_pct(all_pairs, "effective_rent")
within_max_gap = all_pairs["gap_years"] < MAX_GAP_YEARS
reference_pairs = all_pairs[within_max_gap & (all_pairs["gap_years"] > MIN_GAP_YEARS)]
with_six_month_pairs = all_pairs[within_max_gap & (all_pairs["gap_years"] > SIX_MONTH_GAP_YEARS)]


def trimmed_mean(values):
    low, high = values.quantile([TRIM_SHARE, 1 - TRIM_SHARE])
    return values[values.between(low, high)].mean()


by_year = lambda pair_df: pair_df.groupby("lease_start_year")
method_sensitivity = pd.DataFrame({
    "référence (médiane, annualisée, écart > 0,5 an)": by_year(reference_pairs)["annualized_effective_growth_pct"].median(),
    "inclure les baux de 6 mois": by_year(with_six_month_pairs)["annualized_effective_growth_pct"].median(),
    "sans annualisation": by_year(reference_pairs)["raw_effective_growth_pct"].median(),
    "moyenne au lieu de médiane": by_year(reference_pairs)["annualized_effective_growth_pct"].mean(),
    "moyenne tronquée": by_year(reference_pairs)["annualized_effective_growth_pct"].apply(trimmed_mean),
}).loc[REQUIRED_BACKTEST_YEARS].T
method_sensitivity["écart max vs référence"] = method_sensitivity.sub(method_sensitivity.iloc[0]).abs().max(axis=1)
method_sensitivity.round(2)

**Lecture :** aucune variante ne s'écarte de plus d'environ un quart de point de la référence : **la mesure est robuste** aux
choix de filtre, d'annualisation et de statistique. La moyenne simple est un peu plus haute, tirée par quelques fortes hausses,
ce qui justifie la médiane.

## 4. Concessions : loyer contractuel ou loyer effectif?

Pour chaque paire, le mécanisme est exact :

$$\frac{\text{effectif}_t}{\text{effectif}_{t-1}} = \frac{\text{contractuel}_t}{\text{contractuel}_{t-1}} \times \frac{1 - \text{rabais}_t}{1 - \text{rabais}_{t-1}}$$

Un rabais **stable** ne crée aucun écart entre les deux croissances; seul un rabais **qui grossit** d'un bail à l'autre en crée un.
On mesure donc la fréquence des concessions, leur taille, et surtout leur **variation sur les mêmes appartements**.

In [ ]:
leases_in_window = leases[leases["lease_start_year"].between(FIRST_ANALYSIS_YEAR, LAST_OBSERVED_YEAR)]
leases_in_window = leases_in_window.assign(
    discount_pct=discount_pct(leases_in_window["effective_rent"], leases_in_window["contract_rent"]))
concessions_by_year = pd.DataFrame({
    "pct_leases_with_concession": leases_in_window.groupby("lease_start_year")["has_concession"].mean() * PCT,
    "mean_discount_pct": leases_in_window.groupby("lease_start_year")["discount_pct"].mean(),
    "contract_growth": candidate_definitions["same_unit_contract"],
    "effective_growth": candidate_definitions["same_unit_effective"],
    "median_discount_change_pts": median_by_year(pairs, "discount_change_pts"),
    "pct_pairs_discount_up": pairs.groupby("lease_start_year")["discount_change_pts"].apply(lambda change: (change > 0).mean() * PCT),
})
concessions_by_year["growth_gap_pts"] = concessions_by_year["contract_growth"] - concessions_by_year["effective_growth"]
print(concessions_by_year.round(2), end="\n\n")

discount_change_by_type = (pairs[pairs["lease_start_year"] >= REQUIRED_BACKTEST_YEARS[0]]
                           .pivot_table(index="lease_start_year", columns="is_renewal", values="discount_change_pts", aggfunc="median")
                           .rename(columns={RENEWAL: "renouvellement", TURNOVER: "relocation"}))
print("Hausse médiane du rabais (points), par type de bail :")
print(discount_change_by_type.round(2))

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
concessions_by_year[["contract_growth", "effective_growth"]].plot(ax=axes[0], marker="o")
axes[0].legend(["loyer contractuel", "loyer effectif"])
axes[0].set(xlabel="année du nouveau bail", ylabel="croissance médiane (%)", title="Mêmes appartements, deux loyers")
concessions_by_year[["growth_gap_pts", "median_discount_change_pts"]].plot(ax=axes[1], marker="o")
axes[1].legend(["écart contractuel - effectif (points)", "hausse médiane du rabais (points)"])
axes[1].set(xlabel="année du nouveau bail", ylabel="points", title="L'écart suit la hausse du rabais d'un bail à l'autre")
for ax in axes:
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

**Lecture : pourquoi les deux croissances divergent à partir de 2024.**

1. **Les concessions deviennent à la fois plus fréquentes et plus généreuses** : la part des baux avec concession passe de 44-57 %
   (2019-2023) à 73 % en 2024 puis 85 % en 2025, et le rabais moyen de 2-4 % à 9,1 % du loyer contractuel.
2. **Sur les mêmes appartements, le rabais grossit d'un bail à l'autre** : hausse médiane nulle en 2023, puis +1,2 point en 2024
   et +3,0 points en 2025 (le rabais augmente pour 86 % des paires de 2025). C'est presque exactement l'écart entre les deux
   croissances : 1,6 point en 2024 et 3,5 points en 2025 (courbe de droite).
3. **Renouvellements et relocations sont touchés de la même façon** : les concessions servent autant à **retenir** les locataires
   en place qu'à en attirer de nouveaux. Elles se comportent comme une **politique commerciale globale**, présente dans tous les
   immeubles, anciens compris.

En résumé : depuis 2024, Équinoxe continue d'augmenter les loyers inscrits aux baux (+7,0 % en 2025), mais en redonne une part
croissante sous forme de concessions; le loyer perçu ne monte que de 3,6 %.

**Conséquences :** (1) on prévoit le **loyer effectif** (section 1); (2) la **taille du rabais au bail précédent** devient une
variable clé du modèle : un appartement loué avec un gros rabais peut voir ce rabais reculer au bail suivant, et inversement (section 7).

## 5. Renouvellements, relocations et règles provinciales

Le type de bail est celui du **nouveau** bail de chaque paire (`is_renewal`). Les règles diffèrent :

| Cas | Québec (5 immeubles) | Ontario (The Met) |
|---|---|---|
| Renouvellement | hausse encadrée par le **TAL** : taux indicatif annuel, qui ne s'impose que si le locataire conteste | **ligne directrice** provinciale, sauf pour les logements occupés pour la première fois après le 15 novembre 2018, **exemptés** du plafond |
| Relocation | loyer de marché (avec la clause G du bail) | loyer de marché, sans plafond |

**On n'applique jamais le plafond d'une province à l'autre**, et on n'écrête aucune hausse : on mesure ce qui a été signé.

In [ ]:
SEGMENT_LABELS = {"renewal_QC": "renouvellement Québec", "renewal_ON": "renouvellement Ontario", "turnover": "relocation"}
recent_pairs = pairs[pairs["lease_start_year"] >= EXTENDED_BACKTEST_YEARS[0]]
by_segment = recent_pairs.pivot_table(index="lease_start_year", columns="segment", aggfunc="median",
                                      values=["contract_rent_growth_pct", "effective_rent_growth_pct"])
segment_counts = recent_pairs.pivot_table(index="lease_start_year", columns="segment", values="effective_rent_growth_pct", aggfunc="size")
print("Paires par segment :")
print(segment_counts.fillna(0).astype(int), end="\n\n")
print("Croissance médiane (%) :")
print(by_segment.round(2))

### 5a. Renouvellements québécois et taux du TAL

Si les renouvellements suivaient le TAL, la « prime » (hausse contractuelle observée - taux du TAL) serait stable.

In [ ]:
regulatory_rates = pd.read_csv(REGULATORY_RATES_PATH).set_index("year")
renewal_vs_regulation = pd.DataFrame({
    "renewal_QC_contract_growth": by_segment[("contract_rent_growth_pct", "renewal_QC")],
    "tal_rate": regulatory_rates["tal_unheated_pct"],
    "renewal_ON_contract_growth": by_segment[("contract_rent_growth_pct", "renewal_ON")],
    "ontario_guideline": regulatory_rates["ontario_guideline_pct"],
}).loc[EXTENDED_BACKTEST_YEARS]
renewal_vs_regulation["premium_over_tal_pts"] = (renewal_vs_regulation["renewal_QC_contract_growth"]
                                                 - renewal_vs_regulation["tal_rate"])
renewal_vs_regulation["premium_over_guideline_pts"] = (renewal_vs_regulation["renewal_ON_contract_growth"]
                                                       - renewal_vs_regulation["ontario_guideline"])
renewal_vs_regulation.round(2)

**Lecture (Québec) :** la prime est **instable** : environ +4 points en 2021-2022 (TAL très bas, Équinoxe augmente bien au-dessus),
puis proche de 0 depuis 2023. Depuis que le TAL est élevé, les renouvellements québécois le suivent à peu près; avant, ils l'ignoraient.
Le taux du TAL n'est qu'indicatif : on ne l'impose donc pas comme plafond, il sert de contexte réglementaire; la régression retenue ne l’utilise pas comme feature.

### 5b. The Met : la ligne directrice ontarienne s'applique-t-elle?

In [ ]:
the_met_leases = leases[leases["sState"] == ONTARIO]
the_met_renewals = pairs[pairs["segment"] == "renewal_ON"]
pd.Series({
    "premier bail observé à The Met": f"{the_met_leases['lease_start'].min():%Y-%m-%d}",
    "date d'exemption ontarienne (première occupation après)": f"{ONTARIO_EXEMPTION_DATE:%Y-%m-%d}",
    "renouvellements appariés à The Met": len(the_met_renewals),
    "part au-dessus de la ligne directrice de l'année (%)": round(
        (the_met_renewals["contract_rent_growth_pct"]
         > the_met_renewals["lease_start_year"].map(regulatory_rates["ontario_guideline_pct"])).mean() * PCT, 1),
}).to_frame("valeur")

**Lecture (Ontario) :** The Met génère ses premiers baux en 2023, bien après le 15 novembre 2018 : il est **vraisemblablement
exempté** de la ligne directrice. Les données vont dans ce sens : **81 % de ses renouvellements dépassent la ligne directrice de
l'année** (2,5 %), et la hausse contractuelle médiane des renouvellements atteint 6,3 % en 2025, ce qui serait impossible pour un
logement assujetti. Le statut exact reste à confirmer avec un document (date de première occupation).

**Conséquences pour la prévision :**
- les renouvellements et les relocations sont **séparés** dans le modèle (`is_renewal`), et les scénarios sont pondérés par les parts observées en 2025 dans chaque marché;
- le taux réglementaire est **propre à chaque province** : TAL pour les immeubles québécois, ligne directrice pour The Met;
  il reste un contexte d’interprétation; il n’est ni une feature ni un plafond imposé dans la régression;
- The Met n'a des paires qu'à partir de 2024 : sa prévision repose en partie sur ce que le modèle apprend du Québec (section 8).

Écart entre types de baux : en 2022-2023 les renouvellements montent plus vite, en 2024-2025 ce sont les relocations
(loyers de marché en hausse), ce qui justifie de ne pas les fusionner sans le dire au modèle.

## 6. Données publiques : intégrer et réconcilier

Nous retenons trois facteurs publics pour compléter l'historique privé. Ils décrivent les conditions du marché et sont joints aux baux par marché et année.

| Source | Variable retenue | Marché | Période | Interprétation |
|---|---|---|---|---|
| SCHL, Enquête sur les logements locatifs | `vacancy_rate` | RMR Montréal et Ottawa-Gatineau, partie Ontario | Octobre 2021–2025 | pression concurrentielle du marché locatif |
| Statistique Canada, IPC | `general_cpi` | Canada | Moyennes annuelles 2021–2025 | évolution du niveau général des prix, indice 2002=100 |
| Statistique Canada, population active | `unemployement_rate` | Montréal; Toronto comme substitut pour Ottawa | Moyennes annuelles 2021–2025 | contexte du marché du travail, taux en % |

L'inoccupation régionale ne mesure pas celle d'Équinoxe. L'IPC général est un **indice**, pas un taux d'inflation ni une composante loyers. Toronto est un substitut explicite : son chômage n'est pas celui d'Ottawa.

### 6a. Préparer les facteurs retenus

Les transformations restent en mémoire. On conserve les périodes et géographies sources pour vérifier les jointures.

In [ ]:
from pathlib import Path
import re
import unicodedata
import numpy as np
import pandas as pd
from IPython.display import display

RAW_PUBLIC = Path("raw_public_data")
PRIVATE = Path("private_data")
pd.set_option("display.max_columns", 80)
MARKETS = ["Montréal", "Ottawa — partie Ontario"]

def slug(value):
    text = unicodedata.normalize("NFKD", str(value)).encode("ascii", "ignore").decode().lower()
    return re.sub(r"[^a-z0-9]+", "_", text).strip("_")

def read_public_csv(path):
    with path.open(encoding="utf-8-sig") as stream:
        sep = ";" if ";" in stream.readline() else ","
    return pd.read_csv(path, sep=sep, encoding="utf-8-sig").rename(columns={
        "PÉRIODE DE RÉFÉRENCE": "period", "REF_DATE": "period",
        "GÉO": "geo", "GEO": "geo", "VALEUR": "value", "VALUE": "value",
        "UNITÉ DE MESURE": "unit", "UOM": "unit",
        "FACTEUR SCALAIRE": "scalar", "SCALAR_FACTOR": "scalar",
    })

specs = [
    ("IPC.csv", "cpi", "Produits et groupes de produits"),
    ("msrche-du-travail.csv", "labour", "Caractéristiques de la population active"),
]
frames = {}
for filename, family, dimension in specs:
    frame = read_public_csv(RAW_PUBLIC / filename)
    frame["value"] = pd.to_numeric(frame.value, errors="raise")
    frame["Year"] = pd.to_datetime(frame.period, format="%Y-%m").dt.year
    frames[family] = frame


In [ ]:
cmhc, cmhc_sources = extract_public_data(RAW_PUBLIC)
public_columns = ["general_cpi", "vacancy_rate", "unemployement_rate"]
df_public = cmhc.rename(columns={"market": "Market", "year": "Year"}).copy()

cpi = frames["cpi"].loc[frames["cpi"].geo.eq("Canada") & frames["cpi"]["Produits et groupes de produits"].eq("Ensemble")].copy()
assert not cpi.duplicated(["Year", "period"]).any()
assert cpi.unit.eq("2002=100").all()
cpi_yearly = cpi.groupby("Year").value.mean().rename("general_cpi").reset_index()
df_public = df_public.merge(cpi_yearly, on="Year", how="left", validate="many_to_one")

labour = frames["labour"].loc[frames["labour"]["Caractéristiques de la population active"].eq("Taux de chômage")].copy()
labour["Market"] = labour.geo.map({"Montréal, Québec": "Montréal", "Toronto, Ontario": "Ottawa — partie Ontario"})
labour = labour.loc[labour.Market.notna()]
assert not labour.duplicated(["Market", "period"]).any()
assert labour.unit.eq("Pourcent").all()
unemployment = labour.groupby(["Market", "Year"]).value.mean().rename("unemployement_rate").reset_index()
df_public = df_public.merge(unemployment, on=["Market", "Year"], how="left", validate="one_to_one")
df_public = df_public[["Market", "Year"] + public_columns].sort_values(["Market", "Year"]).reset_index(drop=True)
assert not df_public.duplicated(["Market", "Year"]).any()
assert df_public[public_columns].notna().all().all()

source_rows = []
for _, row in df_public[["Market", "Year"]].iterrows():
    year, market = row.Year, row.Market
    for family, frame in [("general_cpi", cpi.loc[cpi.Year.eq(year)]),
                          ("unemployement_rate", labour.loc[labour.Year.eq(year) & labour.Market.eq(market)])]:
        source_rows.append({"Market": market, "Year": year, "factor": family,
                            "source_geo": ", ".join(frame.geo.unique()),
                            "months_available": frame.period.nunique(),
                            "reference_periods": ", ".join(sorted(frame.period.unique()))})
display(df_public)


### 6b. Réconcilier le marché et la tendance interne

Nous comparons la croissance **effective** des mêmes appartements avec les trois facteurs publics. Les graphiques sont descriptifs : une médiane historique résume les paires, tandis que l'IPC et les taux régionaux décrivent un contexte plus large.

La SCHL observe octobre, les autres séries sont des moyennes annuelles, et les baux sont classés selon leur date de début. Ces périodes ne coïncident pas parfaitement. Les échelles sont également différentes : un indice de prix ne doit pas être tracé comme s'il était une hausse en %.

In [ ]:
reconciliation = pairs.assign(Market=pairs.sState.map({QUEBEC: "Montréal", ONTARIO: "Ottawa — partie Ontario"}), Year=pairs.lease_start_year).groupby(["Market", "Year"]).agg(
    nombre_paires=("effective_rent_growth_pct", "size"),
    hausse_effective_mediane_pct=("effective_rent_growth_pct", "median"),
).reset_index().merge(df_public, on=["Market", "Year"], how="left", validate="one_to_one")
display(reconciliation.loc[reconciliation.Year.ge(2021)].round(3))
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for row, market in enumerate(MARKETS):
    internal = reconciliation.loc[reconciliation.Market.eq(market) & reconciliation.Year.ge(2021)]
    external = df_public.loc[df_public.Market.eq(market)]
    for ax, (factor, label) in zip(axes[row], [
        ("vacancy_rate", "Inoccupation (%)"), ("general_cpi", "IPC général (2002=100)"),
        ("unemployement_rate", "Chômage (%)"),
    ]):
        ax.plot(internal.Year, internal.hausse_effective_mediane_pct, "o-", color="tab:blue", label="Équinoxe : hausse effective médiane")
        other = ax.twinx()
        other.plot(external.Year, external[factor], "s--", color="tab:orange", label=label)
        ax.set(title=market, xlabel="Année", ylabel="Hausse effective (%)")
        other.set_ylabel(label)
        ax.set_xticks(range(2021, 2026)); ax.grid(alpha=0.3)
        if row == 0:
            handles1, labels1 = ax.get_legend_handles_labels(); handles2, labels2 = other.get_legend_handles_labels()
            ax.legend(handles1 + handles2, labels1 + labels2, fontsize=7, loc="best")
fig.tight_layout(); plt.show()

**Lecture :** l'inoccupation se détend après 2023 dans les deux marchés. Cela est compatible avec une concurrence accrue et les promotions analysées en section 4, mais ne démontre pas qu'elle les provoque. La hausse effective d'Équinoxe reste influencée par les concessions et ne suit pas nécessairement l'IPC général ou le chômage d'une année à l'autre.

**Conséquence :** nous conservons **uniquement `vacancy_rate`, `general_cpi` et `unemployement_rate` comme attributs publics du modèle**. Le loyer moyen, la rotation et l'univers locatif ne sont pas des features de la version retenue. Les données démographiques et de construction ne sont pas nécessaires à cette spécification. Les règles du TAL et de l'Ontario restent un contexte d'interprétation, pas des prédicteurs de cette régression.

Ce choix limite le nombre de facteurs avec un historique court. Il ne résulte pas d'une démonstration que ces trois facteurs causent les hausses. Pour prévoir l'année *t*, nous utilisons leur valeur de *t − 1*; les dates de publication et révisions devront aussi être vérifiées pour une simulation strictement historique.

## 7. Modèle retenu et résultats

### 7a. Données utilisées

**Y :** `effective_growth_pct`, la hausse annualisée du loyer effectif à unité constante.

| Attributs | Variables retenues |
|---|---|
| Privés | loyer effectif précédent, écart entre les baux, nombre de chambres, marché, type de bail |
| Publics | `general_cpi`, `vacancy_rate`, `unemployement_rate` |

Les facteurs publics sont décalés d'un an. Les clés identifient les appartements; le loyer courant et les concessions du nouveau bail ne sont pas des features.

In [ ]:
# Méthode et renommages repris de unite_constante.ipynb.
UNIT_KEY = ["sPropCode", "sUnitCode"]
MIN_GAP_YEARS = 0.5
MAX_GAP_YEARS = 2.5
DAYS_PER_YEAR = 365.25
PCT = 100
PREVIOUS_LEASE_COLUMNS = ["contract_rent", "effective_rent", "lease_start", "lease_term_months"]
LEASE_COLUMN_RENAMES = {
    "sRent": "contract_rent",
    "sRentEffective": "effective_rent",
    "sConcession": "has_concession",
    "sRenewal": "is_renewal",
    "sTermSeq": "unit_lease_rank",
    "sTermMonths": "lease_term_months",
    "sLeaseFrom": "lease_start",
    "sLeaseTo": "lease_end",
    "sSignDate": "sign_date",
    "sAvailable": "lease_start_duplicate",
    "sLeaseTerm": "lease_term_label_unreliable",
}

def pair_consecutive_leases(lease_df, unit_key):
    """Associe chaque bail au bail précédent de la même unité (colonnes préfixées par `prev_`).

    Retourne uniquement les baux qui ont un bail précédent.
    """
    ordered = lease_df.sort_values(unit_key + ["lease_start"]).copy()
    previous = ordered.groupby(unit_key)[PREVIOUS_LEASE_COLUMNS].shift(1).add_prefix("prev_")
    paired = pd.concat([ordered, previous], axis=1).dropna(subset=["prev_lease_start"])
    paired["gap_years"] = (paired["lease_start"] - paired["prev_lease_start"]).dt.days / DAYS_PER_YEAR
    return paired

def filter_comparable_pairs(paired, price_col, verbose=False):
    """Garde les paires dont l'écart est comparable et dont les deux loyers sont positifs."""
    prev_price_col = f"prev_{price_col}"
    steps = [
        ("écart > MIN_GAP_YEARS", paired["gap_years"] > MIN_GAP_YEARS),
        ("écart < MAX_GAP_YEARS", paired["gap_years"] < MAX_GAP_YEARS),
        ("loyers positifs", (paired[prev_price_col] > 0) & (paired[price_col] > 0)),
    ]
    keep = pd.Series(True, index=paired.index)
    funnel = [("paires appariées", len(paired))]
    for label, condition in steps:
        keep &= condition
        funnel.append((label, int(keep.sum())))
    if verbose:
        funnel_table = pd.DataFrame(funnel, columns=["étape", "paires restantes"])
        funnel_table["paires retirées"] = -funnel_table["paires restantes"].diff().fillna(0).astype(int)
        print(funnel_table.to_string(index=False))
    return paired[keep].copy()

def annualized_growth_pct(paired, price_col):
    """Croissance annualisée (%) entre le bail précédent et le bail courant."""
    price_ratio = paired[price_col] / paired[f"prev_{price_col}"]
    return (price_ratio ** (1 / paired["gap_years"]) - 1) * PCT

def same_unit_growth(lease_df, price_col):
    """Croissance annualisée du loyer à unité constante : apparier, filtrer, annualiser."""
    paired = pair_consecutive_leases(lease_df, UNIT_KEY)
    comparable = filter_comparable_pairs(paired, price_col)
    comparable["growth_pct"] = annualized_growth_pct(comparable, price_col)
    return comparable

def median_growth_table(pairs, by):
    return (pairs.groupby(by)["growth_pct"]
            .agg(pairs="size", median="median")
            .round(2))

private_leases = pd.read_csv("private_data/equinoxe_lease_history.csv", dtype={"sPropCode": str, "sUnitCode": str})
for col in ["sLeaseFrom", "sLeaseTo", "sSignDate", "sAvailable"]:
    private_leases[col] = pd.to_datetime(private_leases[col], errors="raise")
private_leases = private_leases.rename(columns=LEASE_COLUMN_RENAMES)
for col in ["contract_rent", "effective_rent", "lease_term_months", "is_renewal"]:
    private_leases[col] = pd.to_numeric(private_leases[col], errors="raise")
private_leases["lease_start_year"] = private_leases.lease_start.dt.year
assert private_leases[UNIT_KEY].notna().all().all()
assert not private_leases.duplicated(UNIT_KEY + ["lease_start"]).any(), "Baux ambigus : vérifier les doublons avant appariement."

# La cible retenue est la croissance du loyer effectif, comme dans effective_pairs.
effective_pairs = same_unit_growth(private_leases, "effective_rent")
contract_pairs = same_unit_growth(private_leases, "contract_rent")
df_private = effective_pairs.copy()
df_private["Market"] = df_private.sState.map({"Quebec": "Montréal", "Ontario": "Ottawa — partie Ontario"})
df_private["Year"] = df_private.lease_start_year.astype("Int64")
df_private["type_bail"] = df_private.is_renewal.map({0: "Relocation", 1: "Renouvellement"})
df_private["effective_growth_pct"] = df_private.growth_pct
# La hausse contractuelle est un diagnostic complémentaire sur ces mêmes paires.
contract_valid = df_private.contract_rent.gt(0) & df_private.prev_contract_rent.gt(0)
df_private["contract_growth_pct"] = annualized_growth_pct(df_private, "contract_rent").where(contract_valid)
leading = ["sPropCode", "sUnitCode", "Market", "Year", "type_bail", "effective_rent", "prev_effective_rent", "gap_years", "effective_growth_pct"]
df_private = df_private[leading + [c for c in df_private if c not in leading]]
assert np.isfinite(df_private.effective_growth_pct).all()
assert not df_public.duplicated(["Market", "Year"]).any()

private_attributes = [
    "sPropCode", "sUnitCode", "Market", "Year", "type_bail",
    "effective_rent", "prev_effective_rent", "gap_years", "effective_growth_pct",
    "has_concession", "bedrooms",
]
public_attributes = ["general_cpi", "vacancy_rate", "unemployement_rate"]
df_private_details = df_private.copy()
df_private_details["bedrooms"] = pd.to_numeric(df_private_details.sBeds, errors="raise")
df_private = df_private_details[private_attributes].copy()
df_public_retained = df_public[["Year", "Market"] + public_attributes].copy()
assert df_private.shape[1] == 11
assert not df_public_retained.duplicated(["Year", "Market"]).any()
df_final = df_private.merge(
    df_public_retained,
    on=["Year", "Market"],
    how="left",
    validate="many_to_one",
)
assert len(df_final) == len(df_private), "La jointure a multiplié les paires."
assert df_final.columns.tolist() == private_attributes + public_attributes


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

TARGET = "effective_growth_pct"
public_features = ["general_cpi", "vacancy_rate", "unemployement_rate"]
# Repartir des attributs privés de df_final, puis rattacher les facteurs à Year - 1.
regression_data = df_final.drop(columns=public_features).copy()
public_lagged = df_public[["Market", "Year"] + public_features].copy()
public_lagged["public_source_year"] = public_lagged.Year
public_lagged["Year"] = public_lagged.Year + 1
public_lagged = public_lagged.rename(columns={c: "lag1_" + c for c in public_features})
regression_data = regression_data.merge(public_lagged, on=["Market", "Year"], how="left", validate="many_to_one")
NUMERIC_FEATURES = ["prev_effective_rent", "gap_years", "bedrooms"] + ["lag1_" + c for c in public_features]
CATEGORICAL_FEATURES = ["Market", "type_bail"]
FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES
eligible = (regression_data[FEATURES + [TARGET]].notna().all(axis=1)
            & np.isfinite(regression_data[NUMERIC_FEATURES + [TARGET]].astype(float)).all(axis=1))
model_data = regression_data.loc[eligible].copy()



TEST_YEAR = 2025
train = model_data.loc[model_data.Year.lt(TEST_YEAR)].copy()
test = model_data.loc[model_data.Year.eq(TEST_YEAR)].copy()
assert not train.empty and not test.empty
assert train.Year.max() < test.Year.min()
assert set(test.Market).issubset(set(train.Market))

# Ne pas inclure Year : le modèle utilise les facteurs économiques et le marché.
def make_linear_model():
    preprocessing = ColumnTransformer([
        ("numeric", StandardScaler(), NUMERIC_FEATURES),
        ("category", OneHotEncoder(drop="first", handle_unknown="error", sparse_output=False), CATEGORICAL_FEATURES),
    ])
    return Pipeline([("preprocessing", preprocessing), ("regression", LinearRegression())])

linear_model = make_linear_model()
linear_model.fit(train[FEATURES], train[TARGET])
train_prediction = linear_model.predict(train[FEATURES])
test_prediction = linear_model.predict(test[FEATURES])
# Référence simple calculée sur l'entraînement uniquement.
baseline = np.full(len(test), train[TARGET].mean())
validation_results = pd.DataFrame([
    {"modele": "Régression linéaire", "MAE_points": mean_absolute_error(test[TARGET], test_prediction),
     "RMSE_points": np.sqrt(mean_squared_error(test[TARGET], test_prediction))},
    {"modele": "Moyenne historique", "MAE_points": mean_absolute_error(test[TARGET], baseline),
     "RMSE_points": np.sqrt(mean_squared_error(test[TARGET], baseline)), "R2": r2_score(test[TARGET], baseline)},
]).set_index("modele")

df_predictions = test[["sPropCode", "sUnitCode", "Market", "Year", "type_bail", TARGET]].copy()
df_predictions["predicted_growth_pct"] = test_prediction
df_predictions["error_points"] = df_predictions.predicted_growth_pct - df_predictions[TARGET]


In [ ]:
# Dataframe final transmis au modèle : clés, features et cible.
df_modele_final = model_data[["sPropCode", "sUnitCode", "Year"] + FEATURES + [TARGET]].copy()
print(f"{len(df_modele_final):,} paires utilisables; test : {len(test):,} paires en 2025.")
display(df_modele_final.head(10).round(3))

### 7b. Comparer les modèles sur 2025

Entraînement avant 2025, test sur 2025. **MAE et RMSE : plus faibles = mieux.** Les erreurs sont en points de pourcentage.

In [ ]:
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.impute import SimpleImputer

def tree_pipeline(estimator):
    return Pipeline([
        ("preprocessing", ColumnTransformer([
            ("numeric", SimpleImputer(strategy="median"), NUMERIC_FEATURES),
            ("category", OneHotEncoder(categories=[MARKETS, ["Relocation", "Renouvellement"]], drop="first", handle_unknown="error", sparse_output=False), CATEGORICAL_FEATURES),
        ])),
        ("regression", estimator),
    ])
comparison_models = {
    "Random Forest": tree_pipeline(RandomForestRegressor(n_estimators=300, max_depth=8, min_samples_leaf=20, random_state=42, n_jobs=1)),
    "Gradient Boosting": tree_pipeline(GradientBoostingRegressor(n_estimators=200, learning_rate=0.03, max_depth=2, min_samples_leaf=20, loss="squared_error", random_state=42)),
}
comparison_results = validation_results[["MAE_points", "RMSE_points"]].copy()
for name, fitted in comparison_models.items():
    fitted.fit(train[FEATURES], train[TARGET])
    prediction = fitted.predict(test[FEATURES])
    comparison_results.loc[name] = [
        mean_absolute_error(test[TARGET], prediction),
        np.sqrt(mean_squared_error(test[TARGET], prediction)),
    ]
display(comparison_results.sort_values("MAE_points").round(4))

**Choix : régression linéaire.** Elle obtient la meilleure MAE (**4,22 points**) et la meilleure RMSE (**5,97 points**), devant le Gradient Boosting et le Random Forest. Elle améliore légèrement la référence historique (MAE **4,35 points**). Le gain reste faible : ce choix est relatif aux modèles testés, pas une garantie de forte précision.

### 7c. Backtests 2023–2025 et prévision 2026

Chaque backtest entraîne uniquement avant l'année cible. Le tableau distingue l'erreur sur la **moyenne annuelle** des erreurs par **paire de baux**.

Le modèle final est réentraîné jusqu'en 2025. Il utilise les facteurs publics de 2025, un intervalle d'un an et les proportions de renouvellements/relocations de 2025 par marché.

In [ ]:
final_training_data = model_data.loc[model_data.Year.le(2025)].copy()
final_model = make_linear_model()
final_model.fit(final_training_data[FEATURES], final_training_data[TARGET])

# Tous les appartements ayant un bail connu, pas uniquement ceux avec une paire récente.
CUTOFF = pd.Timestamp("2025-12-31")
known_leases = private_leases.loc[private_leases.lease_start.le(CUTOFF)].copy()
latest = known_leases.sort_values(UNIT_KEY + ["lease_start"]).groupby(UNIT_KEY, as_index=False).tail(1).copy()
latest["Market"] = latest.sState.map({"Quebec": "Montréal", "Ontario": "Ottawa — partie Ontario"})
latest = latest.loc[latest.effective_rent.gt(0) & np.isfinite(latest.effective_rent) & latest.Market.notna()].copy()
assert not latest.duplicated(UNIT_KEY).any()

# Proportions issues des paires 2025, calculées séparément par marché.
recent = df_final.loc[df_final.Year.eq(2025) & df_final.type_bail.isin(["Relocation", "Renouvellement"])]
relocation_share_2026 = recent.assign(relocation=recent.type_bail.eq("Relocation")).groupby("Market").relocation.mean().to_dict()
# Pour modifier l'hypothèse : relocation_share_2026["Montréal"] = 0.30

scenario_parts = []
for kind in ["Renouvellement", "Relocation"]:
    scenario = latest[UNIT_KEY + ["sBuilding", "Market", "effective_rent", "lease_start", "sBeds"]].copy()
    scenario = scenario.rename(columns={"effective_rent": "prev_effective_rent", "lease_start": "reference_lease_start", "sBeds": "bedrooms"})
    scenario["Year"] = FORECAST_YEAR
    scenario["gap_years"] = 1.0
    scenario["type_bail"] = kind
    scenario_parts.append(scenario)
scenarios_2026 = pd.concat(scenario_parts, ignore_index=True)
source_2025 = df_public.loc[df_public.Year.eq(2025), ["Market"] + public_features].rename(columns={c: "lag1_" + c for c in public_features})
scenarios_2026 = scenarios_2026.merge(source_2025, on="Market", how="left", validate="many_to_one")
assert scenarios_2026[FEATURES].notna().all().all()
assert len(scenarios_2026) == 2 * len(latest)
scenarios_2026["predicted_growth_pct"] = final_model.predict(scenarios_2026[FEATURES])
scenarios_2026["predicted_effective_rent"] = scenarios_2026.prev_effective_rent * (
    1 + scenarios_2026.predicted_growth_pct / 100
) ** scenarios_2026.gap_years
assert scenarios_2026.predicted_effective_rent.gt(0).all()

def combine_2026_scenarios(scenarios, relocation_shares):
    result = scenarios.copy()
    shares = result.Market.map(relocation_shares)
    if shares.isna().any() or not shares.between(0, 1).all():
        raise ValueError("Chaque marché doit avoir une proportion de relocations entre 0 et 1.")
    result["scenario_weight"] = np.where(result.type_bail.eq("Relocation"), shares, 1 - shares)
    result["weighted_growth"] = result.predicted_growth_pct * result.scenario_weight
    unit_forecasts = result.groupby(UNIT_KEY + ["Market", "sBuilding"], as_index=False).agg(
        predicted_growth_pct=("weighted_growth", "sum"), total_weight=("scenario_weight", "sum")
    )
    assert np.allclose(unit_forecasts.total_weight, 1)
    return unit_forecasts.drop(columns="total_weight")

def estimate_2026(relocation_shares=None):
    """Hausse effective moyenne prévue en 2026 (%) pour les appartements de l'extrait."""
    shares = relocation_share_2026 if relocation_shares is None else relocation_shares
    return float(combine_2026_scenarios(scenarios_2026, shares).predicted_growth_pct.mean())

df_forecast_2026 = combine_2026_scenarios(scenarios_2026, relocation_share_2026)
forecast_2026_pct = estimate_2026()

def prepare_template_leases(leases):
    data = leases.copy()
    if "bedrooms" not in data:
        data["bedrooms"] = pd.to_numeric(data.sBeds, errors="raise")
    data = data.rename(columns={k: v for k, v in LEASE_COLUMN_RENAMES.items() if k in data and v not in data})
    data["lease_start"] = pd.to_datetime(data.lease_start)
    data["lease_start_year"] = data.lease_start.dt.year
    data["Market"] = data.sState.map({"Quebec": "Montréal", "Ontario": "Ottawa — partie Ontario"})
    data["Year"] = data.lease_start_year
    data["type_bail"] = data.is_renewal.map({0: "Relocation", 1: "Renouvellement"})
    return data


def prepare_template_pairs(leases, external):
    pairs = same_unit_growth(leases, "effective_rent")
    pairs[TARGET] = pairs.growth_pct
    public = external[["Market", "Year"] + public_features].copy()
    assert not public.duplicated(["Market", "Year"]).any()
    public["Year"] += 1
    public = public.rename(columns={c: "lag1_" + c for c in public_features})
    pairs = pairs.merge(public, on=["Market", "Year"], how="left", validate="many_to_one")
    return pairs.dropna(subset=FEATURES + [TARGET])


def fit_template_model(leases, year, external):
    history = prepare_template_leases(leases)
    history = history.loc[history.lease_start.lt(pd.Timestamp(year=year, month=1, day=1))].copy()
    public_history = external.loc[external.Year.lt(year)].copy()
    training = prepare_template_pairs(history, public_history)
    if training.empty:
        raise ValueError("Pas de paires et de facteurs publics disponibles avant l'année cible.")
    # Catégories définies à l'avance; un marché sans historique est identifié dans le backtest.
    preprocessing = ColumnTransformer([
        ("numeric", StandardScaler(), NUMERIC_FEATURES),
        ("category", OneHotEncoder(categories=[MARKETS, ["Relocation", "Renouvellement"]],
                                   drop="first", handle_unknown="error", sparse_output=False), CATEGORICAL_FEATURES),
    ])
    fitted = Pipeline([("preprocessing", preprocessing), ("regression", LinearRegression())])
    fitted.fit(training[FEATURES], training[TARGET])
    return fitted, history, training, public_history


def estimate_2026(leases, asking, external=None):
    """Hausse effective annualisée moyenne prévue en 2026 (%), mêmes appartements."""
    public = df_public if external is None else external
    fitted, history, training, public_history = fit_template_model(leases, 2026, public)
    latest = history.sort_values(UNIT_KEY + ["lease_start"]).groupby(UNIT_KEY).tail(1).copy()
    latest = latest.loc[latest.effective_rent.gt(0) & latest.Market.notna()]
    parts = []
    # Part de relocations de la dernière année de paires utilisables dans chaque marché.
    shares = {}
    for market, group in training.groupby("Market"):
        recent = group.loc[group.Year.eq(group.Year.max())]
        shares[market] = recent.type_bail.eq("Relocation").mean()
    for kind in ["Renouvellement", "Relocation"]:
        scenario = latest[UNIT_KEY + ["Market", "effective_rent", "bedrooms"]].rename(columns={"effective_rent": "prev_effective_rent"}).copy()
        scenario["gap_years"] = 1.0
        scenario["type_bail"] = kind
        source = public_history.loc[public_history.Year.eq(2025), ["Market"] + public_features].rename(columns={c: "lag1_" + c for c in public_features})
        scenario = scenario.merge(source, on="Market", how="left", validate="many_to_one")
        if scenario[FEATURES].isna().any().any():
            raise ValueError("Facteurs publics 2025 ou caractéristiques de scénario manquants.")
        share = scenario.Market.map(shares)
        if share.isna().any():
            raise ValueError("Aucun historique pour estimer le poids d'un marché.")
        scenario["weight"] = share if kind == "Relocation" else 1 - share
        scenario["weighted_growth"] = fitted.predict(scenario[FEATURES]) * scenario.weight
        parts.append(scenario)
    unit_growth = pd.concat(parts).groupby(UNIT_KEY).weighted_growth.sum()
    return float(unit_growth.mean())


def backtest(leases, target_year):
    """Entraîne avant target_year et compare aux paires réelles de cette année."""
    fitted, history, training, public_history = fit_template_model(leases, target_year, df_public)
    # Les baux de l'année cible ne servent qu'à mesurer Y après l'entraînement.
    evaluation_leases = prepare_template_leases(leases)
    evaluation_leases = evaluation_leases.loc[evaluation_leases.lease_start.lt(pd.Timestamp(year=target_year + 1, month=1, day=1))]
    evaluation = prepare_template_pairs(evaluation_leases, public_history)
    evaluation = evaluation.loc[evaluation.Year.eq(target_year)]
    if evaluation.empty:
        raise ValueError("Aucune paire évaluable pour l'année cible.")
    predicted = fitted.predict(evaluation[FEATURES])
    unseen_markets = sorted(set(evaluation.Market) - set(training.Market))
    return {
        "annee": target_year, "nombre_paires": len(evaluation),
        "hausse_predite_moyenne_pct": float(predicted.mean()),
        "hausse_observee_moyenne_pct": float(evaluation[TARGET].mean()),
        "erreur_moyenne_points": float(predicted.mean() - evaluation[TARGET].mean()),
        "MAE_points": mean_absolute_error(evaluation[TARGET], predicted),
        "RMSE_points": np.sqrt(mean_squared_error(evaluation[TARGET], predicted)),
        "marches_sans_historique": ", ".join(unseen_markets),
    }

asking_model = pd.read_csv("private_data/equinoxe_asking_history.csv", dtype={"sPropCode": str, "sUnitCode": str})
estimation_gabarit_2026 = estimate_2026(private_leases, asking_model, external=df_public)
backtest_results = pd.DataFrame([backtest(private_leases, year) for year in (2023, 2024, 2025)])
assert np.isclose(estimation_gabarit_2026, forecast_2026_pct)


### Coefficients de la régression

Le tableau distingue le modèle de **validation** (entraîné avant 2025) du modèle **final** (entraîné jusqu'en 2025 pour prévoir 2026). Un coefficient positif augmente la hausse prédite; un coefficient négatif la réduit, toutes les autres variables du modèle étant maintenues constantes.

Le coefficient original indique les **points de hausse pour +1 unité** du facteur. Le coefficient standardisé indique les points pour +1 écart-type; pour les catégories, il mesure l'écart à la référence. Ces associations ne sont pas des effets causaux.

In [ ]:
def coefficient_table(fitted):
    preprocessing = fitted.named_steps["preprocessing"]
    regression = fitted.named_steps["regression"]
    scaler = preprocessing.named_transformers_["numeric"]
    encoder = preprocessing.named_transformers_["category"]
    beta = regression.coef_
    rows = [
        {"feature": feature, "coefficient_original": coefficient / scale,
         "coefficient_standardise": coefficient}
        for feature, coefficient, scale in zip(NUMERIC_FEATURES, beta[:len(NUMERIC_FEATURES)], scaler.scale_)
    ]
    rows += [
        {"feature": feature, "coefficient_original": coefficient,
         "coefficient_standardise": coefficient}
        for feature, coefficient in zip(encoder.get_feature_names_out(CATEGORICAL_FEATURES), beta[len(NUMERIC_FEATURES):])
    ]
    intercept = regression.intercept_ - np.sum(beta[:len(NUMERIC_FEATURES)] * scaler.mean_ / scaler.scale_)
    return pd.DataFrame(rows).set_index("feature"), float(intercept)

initial_coefficients, initial_intercept = coefficient_table(linear_model)
final_coefficients, final_intercept = coefficient_table(final_model)
df_coefficients = pd.DataFrame({
    "validation_coefficient_original": initial_coefficients.coefficient_original,
    "final_coefficient_original": final_coefficients.coefficient_original,
    "final_coefficient_standardise": final_coefficients.coefficient_standardise,
})
display(df_coefficients.round(6))
print(f"Constante finale (unités originales) : {final_intercept:.4f}")
encoder = final_model.named_steps["preprocessing"].named_transformers_["category"]
for name, categories in zip(CATEGORICAL_FEATURES, encoder.categories_):
    print(f"Référence pour {name} : {categories[0]}")

matrix = final_model.named_steps["preprocessing"].transform(final_training_data[FEATURES])
design = np.column_stack([np.ones(len(matrix)), matrix])
if np.linalg.matrix_rank(design) < design.shape[1]:
    print("Certains coefficients ne sont pas identifiables séparément avec cet historique court.")


In [ ]:
display(backtest_results.round(4))
annual_backtest_mae = backtest_results.erreur_moyenne_points.abs().mean()
print(f"MAE des moyennes annuelles, backtests 2023–2025 : {annual_backtest_mae:.2f} points.")
print(f"Prévision 2026 : {forecast_2026_pct:.2f} % de hausse effective moyenne.")

**Lecture :** les écarts moyens annuels sont d'environ **+2,44 points en 2023**, **−0,60 en 2024** et **−0,38 en 2025**, soit une MAE annuelle de **1,14 point**. Ottawa n'a pas encore de paires d'entraînement dans le backtest 2024; ses données publiques sont bien disponibles.

**Résultat retenu : +5,44 % en 2026**, moyenne des prévisions par appartement après pondération des deux types de bail. Cette estimation reste conditionnelle aux hypothèses et à un historique court.

## 8. Conclusion

> **Hausse effective moyenne prévue pour 2026 : 5,44 %**, à appartement constant, renouvellements et relocations pondérés. Les scénarios de poids vont de **4,93 %** (tous renouvellements) à **6,24 %** (toutes relocations); ils ne constituent pas un intervalle de confiance.

Nous retenons la régression linéaire parmi les trois modèles testés, tout en rapportant sa faible précision individuelle et les hypothèses qui conditionnent cette estimation.

## 9. Références

**Données publiques** (fichiers dans `raw_public_data/` et `public_data/`, 2021-2025)
- SCHL, *Enquête sur les logements locatifs*, tableaux de données du Rapport sur le marché locatif 2021-2025 (`rmr-canada-AAAA-fr.xlsx`) :
  inoccupation, rotation, loyer moyen, univers, variation à échantillon fixe. https://www.cmhc-schl.gc.ca
- Statistique Canada, tableau 18-10-0004-01, *Indice des prix à la consommation mensuel*, Canada. https://www150.statcan.gc.ca
- Statistique Canada, tableau 14-10-0460-01 (population active, RMR) et 17-10-0149-01 (composantes de l'accroissement démographique, RMR).
- Tribunal administratif du logement, calcul annuel de l'augmentation de loyer. https://www.tal.gouv.qc.ca
- Gouvernement de l'Ontario, ligne directrice sur l'augmentation des loyers et exemption des logements occupés après le
  15 novembre 2018. https://www.ontario.ca/page/rent-increase-guideline

**Méthode et outils**
- JADCO, *Clés en main* (consignes) et `starter.ipynb`.
- scikit-learn, `HistGradientBoostingRegressor`. https://scikit-learn.org
- pandas, NumPy, matplotlib, joblib.

**Outils d'IA**
- Claude Code (Anthropic) : aide à l'écriture du code et des textes, à l'extraction des séries publiques, aux tests de variables et à la mise en forme
  du notebook. Les choix de méthode, les vérifications et les conclusions ont été revus par l'équipe.

**Notebooks de travail** (non évalués, détail des analyses) : `exploration`, `unite_constante`, `concessions`,
`renouvellements_relocations`, `donnees_publiques`, `modele_v1`.